# Getting started with gitrecon

gitrecon explores GitHub activity of users, organizations, repositories and projects:
it **collects** data into a raw buffer, **maps** who touches what and **concludes labels**
for what is happening.

This notebook walks through the core objects with concrete constructor calls.
Everything here is also reachable from the CLI (`gitrecon --help`) - the table at the end
shows the matching commands.

> Run it: `task launch` in this folder (or `task examples:launch` from the repository root).

## 1. Configuration

`Config` holds paths and credentials. Paths are anchored at the project root, so the
notebook works from any directory. `load_env()` reads dotenv files (see
`gitrecon.config.env_files()`) without overriding variables already set.

In [ ]:
from gitrecon.config import Config, load_env

load_env()  # GH_TOKEN / HF_TOKEN from dotenv files, never overriding your shell
config = Config()

In [ ]:
print("data dir:    ", config.data_dir)
print("datasets dir:", config.datasets_dir)
print("token set:   ", bool(config.github_token))

Every field can be given explicitly, e.g. a throwaway data directory for experiments:

In [ ]:
import tempfile
from pathlib import Path

scratch = Config(data_dir=Path(tempfile.mkdtemp(prefix="gitrecon-")))
scratch.raw_dir

## 2. Talking to GitHub: `GitHubClient`

A thin REST client: it sends conditional requests (ETag), follows pagination and waits
when the rate limit is exhausted (`wait_on_limit=True`).

In [ ]:
from gitrecon.sources.github_api import GitHubClient

client = GitHubClient(config=config, wait_on_limit=True)
core = client.get("/rate_limit").data["resources"]["core"]
core

## 3. Models: one class per GitHub structure

API payloads become dataclasses with a stable graph `key`. Each model has `from_api()`.

In [ ]:
from gitrecon.models import Repository, User

user = User.from_api(client.get("/users/sarverott").data)
repo = Repository.from_api(client.get("/repos/Sarverott/gitrecon").data)

print(user.key, "|", user.name, "| public repos:", user.public_repos, "| since", user.created_at.date())
print(repo.key, "|", repo.language, "| stars:", repo.stargazers_count)

Models can also be built by hand - useful in tests and when only a name is known:

In [ ]:
User(login="OctoCat").key, Repository(full_name="Sarverott/gitrecon").owner

## 4. Repositories a user has starred

`stars.starred()` returns `Star` objects (user, repository, when). `max_pages` limits how
many pages of 100 are fetched - a full listing for an active account can be thousands.

In [ ]:
from gitrecon.sources import stars

found = stars.starred(client, "sarverott", max_pages=1)
for star in found[:10]:
    print(f"{star.starred_at:%Y-%m-%d}  {star.repo.full_name:<45} {star.repo.language or '-'}")
print("...", len(found), "on the first page")

## 5. Keeping what was collected: `RawBuffer`

The raw buffer is append-only gzip JSONL, partitioned by source and UTC hour - the same
format GH Archive ships. Analysis is always rebuilt from it.

In [ ]:
from gitrecon.storage import RawBuffer

buffer = RawBuffer(root=scratch.raw_dir)
buffer.append("stars", stars.starred_raw(client, "sarverott", max_pages=1), suffix="user-sarverott")
print(buffer.sources(), [p.name for p in buffer.files("stars")])
next(buffer.read("stars"))["repo"]["full_name"]

## The same from the command line

| Notebook | CLI |
| --- | --- |
| `stars.starred(client, "sarverott")` | `gitrecon stars sarverott` |
| `buffer.append("stars", ...)` | `gitrecon stars sarverott --save` |
| `buffer.sources()` | `gitrecon status` |

Next: [activity-labels](../activity-labels/notebook.ipynb) - from events to labels.